# 0907 6일차

## 0. 파이썬 문법 - 딕셔너리(dict)

`{'키': 값}` 형태로 데이터를 이름표(키)에 연결해 저장하는 자료구조

```python
변수명 = {'키1': 값1, '키2': 값2, ...}
```

**리스트와의 차이**
- 리스트는 `list[0]`처럼 숫자 인덱스로 접근함
- 딕셔너리는 이름표(키)로 접근함

**구성 요소**
1. `{ }` : 딕셔너리를 정의하는 중괄호
2. 키(key) : 데이터의 이름표. 중복 불가, 변경 불가능한 타입만 가능
3. `:` : 키와 값을 연결
4. 값(value) : 모든 타입 가능 (숫자, 문자열, 리스트 등)

### 0-1. 값 조회와 주요 메서드

```python
data = {'loss': [0.5, 0.3], 'val_loss': [0.6, 0.4]}

data['loss']           # [0.5, 0.3]  <- 리스트 반환
data.keys()            # dict_keys(['loss', 'val_loss'])
data.values()          # dict_values([[0.5, 0.3], [0.6, 0.4]])
data.get('accuracy')   # None  <- 없는 키도 KeyError 없이 안전하게
```

**조회 방법**
1. `data['키']` : 없는 키를 찾으면 `KeyError`
2. `data.get('키')` : 없는 키면 `None`을 반환함

### 0-2. History 객체의 `hist.history`

`model.fit()`이 반환하는 History 객체의 저장소(`hist.history`)가 딕셔너리임

```python
hist = model.fit(x_train, y_train, epochs=100, validation_split=0.2)

print(type(hist.history))     # <class 'dict'>
print(hist.history.keys())    # dict_keys(['loss', 'val_loss'])
```

1. `hist.history['loss']` : epoch별 train loss가 담긴 리스트
2. `hist.history['val_loss']` : epoch별 validation loss

- 키로 리스트를 꺼내 `plt.plot()`에 넣어 손실 곡선을 그림 (§6)

## 1. verbose

학습·평가 중 로그를 얼마나 출력할지 정하는 인자. 결과에는 영향을 주지 않음

- 단어 뜻이 "말이 많은"임. 리눅스 `tar -v`, pip `-q`(quiet)와 같은 관례

**값의 종류**
1. `0` : 아무것도 출력하지 않음
2. `1` : 기본값. `Epoch 1/3` + 진행바 + `7/7 ━━━ 0s 3ms/step - loss: 14.27`
3. `2` : `Epoch 1/3` + `7/7 - 0s - 4ms/step - loss: 15.07` (진행바만 빠짐)
4. `3` 이상 : `Epoch 1/3`만

### 1-1. verbose 규칙

1. `fit`, `evaluate`, `predict`에 각각 지정해야 함

```python
model.fit(x_train, y_train, verbose=0)      # 각각
model.evaluate(x_test, y_test, verbose=0)   # 따로
model.predict(x_test, verbose=0)            # 지정해야 함
```

   - 셋 다 기본값이 `1`이라 `fit`만 꺼도 나머지는 계속 출력함
   - `fit(verbose=0)`인데 `1/1 ━━━ - loss: 0.0102`가 출력됐다면 그것은 `evaluate`가 낸 것
   - 배치 개수로 구분할 수 있음. `x_test` 3개면 `1/1`, `x_train` 7개를 `batch_size=1`로 돌면 `7/7`
2. 속도 차이는 1초 미만임

| | 배치 1089개 × 10 epoch | 배치 18개 × 10 epoch |
|---|---|---|
| `verbose=0` | 6.92초 (기준) | 0.42초 |
| `verbose=1` | 7.01초 (+1.3%) | 0.43초 |
| `verbose=2` | 7.10초 (+2.7%) | 0.41초 |

   - 진행바는 매 배치가 아니라 일정 시간 간격으로만 갱신되기 때문임

**값을 고르는 기준 - 로그 가독성**

| 상황 | 값 | 이유 |
|---|---|---|
| 반복 실험 | `0` | 500 epoch이면 로그가 1,000줄이라 R²·RMSE 결과를 찾기 어려움 |
| 지켜볼 때 | `1` | `val_loss` 추이, 발산·`NaN`을 조기에 발견 |
| 파일로 남길 때 | `2` | 진행바는 `\r`로 덮어쓰므로 파일에 `━━━`가 쌓임 |

### 1-2. cf) `verbose=3`이 문서에 없는 이유

```python
# keras/src/callbacks/progbar_logger.py
if self.verbose and self.epochs > 1:      # 0만 아니면 통과 -> Epoch N/M 출력
    io_utils.print_msg(f"Epoch {epoch + 1}/{self.epochs}")
...
if self.verbose == 1:                     # 진행바
```

- 내용을 출력하는 분기는 `== 1`과 `== 2`뿐이라, 3 이상은 두 분기에 걸리지 않고 `Epoch N/M`만 남음
- 공식 문서에는 `"auto", 0, 1, 2` 네 가지만 있음. loss까지 출력하는 `verbose=2`가 상위 호환

## 2. 검증 데이터 (validation)

훈련 중에 과적합 여부를 실시간으로 확인하려고 train에서 떼어놓는 데이터

**검증 데이터가 필요한 이유**
- 3일차 §5-1에서 `fit`의 loss와 `evaluate`의 loss를 비교했지만, 그것은 훈련이 끝난 뒤에야 알 수 있음
- loss만 보면 과적합을 잡을 수 없음 → 데이터를 통째로 외워도 `loss`는 계속 내려감

| loss | val_loss | 진단 |
|---|---|---|
| ↓ | ↓ | 정상 |
| ↓ | **↑ 로 꺾임** | **과적합** → 여기가 멈출 지점 |
| 둘 다 높은 채 정체 | | 과소적합 |

### 2-1. train / validation / test

```
전체 데이터
   ├─ train        가중치를 실제로 학습
   ├─ validation   매 epoch 점검 (훈련에는 안 씀)
   └─ test         훈련이 끝난 뒤 최종 채점
```

| | 언제 | 몇 번 | 쓰는 곳 |
|---|---|---|---|
| **validation** | 훈련 중 | 매 epoch | `fit(validation_data=...)` |
| **test** | 훈련 후 | 한 번 | `evaluate(x_test, y_test)` |

**validation과 test를 나누는 이유**
- validation은 튜닝 기준으로 계속 들여다보는 데이터임
- 층을 바꾸고 epoch을 조절하는 근거로 쓰다 보면 결국 그 데이터에 맞춰짐
- 그래서 한 번도 보지 않은 test를 따로 남겨둠 → 튜닝은 `val_loss`로, `x_test`는 마지막에 한 번만 확인

## 3. 검증 데이터 만드는 방법

3일차 §2의 train/test 나누기와 같은 흐름으로 발전함

**방법의 종류**
1. 값을 직접 적기 : `keras16_validation1.py`
2. 슬라이싱 : `keras16_validation2.py`
3. `train_test_split` 두 번 : `keras16_validation3_train_test.py`
4. `validation_split` : `keras16_validation4_split.py`

### 3-1. 직접 적기 / 슬라이싱

```python
# 1) 6 / 2 / 2
x_train = np.array([1, 2, 3, 4, 5, 6])
x_val   = np.array([7, 8])
x_test  = np.array([9, 10])

model.fit(x_train, y_train, validation_data=(x_val, y_val))

# 2) 8 / 4 / 4
x = np.array(range(1, 17))
x_train = x[:8]        # 1 ~ 8
x_valid = x[8:12]      # 9 ~ 12
x_test  = x[12:]       # 13 ~ 16
```

### 3-2. train_test_split 두 번

```python
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=12, random_state=1)
x_train, x_val,  y_train, y_val  = train_test_split(x_train, y_train, train_size=8, random_state=2)
```

**규칙**
1. 먼저 test를 떼고, 남은 12개를 다시 8/4로 나눔
   - `x_train`을 덮어쓰므로 순서가 중요함
2. `train_size`에 정수를 주면 개수, 소수를 주면 비율
3. 두 호출의 `random_state`는 같을 필요 없음(1, 2도 가능). 값을 주는 것 자체가 중요함

### 3-3. validation_split

`fit`에게 검증 데이터 분리를 맡기는 인자

```python
model.fit(x_train, y_train, batch_size=2, validation_split=0.33)
```

- 미리 나눌 필요 없이 한 줄로 끝남
- 비율은 `x_train` 기준임 (전체의 33%가 아님)

**`validation_split`과 `validation_data`의 차이**

| | `validation_split` | `validation_data` |
|---|---|---|
| 만드는 주체 | `fit` | 사용자 |
| 섞는가 | 안 섞음 (셔플 전 마지막 몫) | 사용자가 정한 대로 |
| 전처리 개입 | 불가 | 가능 |

- 둘 다 주면 `validation_data`가 적용됨

**`validation_data`가 필요한 상황**
1. 정렬된 데이터 : `split`은 뒤에서 자르므로 검증셋이 한 구간에 몰림
2. 시계열 : 반드시 뒤쪽 구간이 검증이어야 함 (3일차 §2-4)
3. 모델 A/B 비교 : 같은 검증셋이어야 같은 기준으로 비교됨
4. 전처리(정규화) : train으로만 기준을 잡고 검증에 적용해야 함

```python
scaler.fit(x_train)                    # 훈련 데이터로만 기준을 잡고
x_val = scaler.transform(x_val)        # 검증엔 그 기준을 적용만
```

- `validation_split`은 `fit` 안에서 나누므로 이 작업을 넣을 자리가 없음

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

x = np.array(range(1, 17))
y = np.array(range(1, 17))

# 슬라이싱 - 순서대로 잘림
print("슬라이싱  train:", x[:8], " val:", x[8:12], " test:", x[12:])

# train_test_split 두 번 - 섞여서 잘림
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=12, random_state=1)
x_train, x_val,  y_train, y_val  = train_test_split(x_train, y_train, train_size=8, random_state=2)
print("split    train:", np.sort(x_train), " val:", np.sort(x_val), " test:", np.sort(x_test))

# random_state를 서로 다르게 줘도(1, 2) 매번 같은 결과가 재현됨

## 4. 실제 데이터셋에 validation_split 적용

`keras17_val1` ~ `keras17_val5`는 기존 코드에 `validation_split` 한 줄만 추가한 것

| 파일 | 데이터셋 | `validation_split` | epochs / batch |
|---|---|---|---|
| `keras17_val1` | 캘리포니아 | `0.2` | 1000 / 10 |
| `keras17_val2` | 당뇨 | `0.33` | 640 / 16 |
| `keras17_val3` | 보스턴 | `0.25` | 1000 / 1 |
| `keras17_val4` | 따릉이 | `0.2` | 500 / 32 |
| `keras17_val5` | 캐글 자전거 | `0.2` | 500 / 8 |

- 비율에 정답은 없고, `0.2` ~ `0.33`을 흔히 씀

### 4-1. 훈련 데이터가 줄어드는 비용

```
따릉이:   x_train 1062 -> 실제 훈련 849 + validation 213
캐글:     x_train 8708 -> 실제 훈련 6966 + validation 1742
```

- 케라스는 내림으로 자름 : `int(1062 × 0.8) = 849`
- 훈련 데이터가 줄어도 쓰는 이유는 과적합이 시작되는 시점을 볼 수 있기 때문임

### 4-2. cf) 같은 인자를 두 번 쓰면 SyntaxError

```python
model.fit(x_train, y_train, epochs=500, batch_size=32, validation_split=0.2,
          validation_split=0.1
          )
```

```
SyntaxError: keyword argument repeated: validation_split
```

- `SyntaxError`는 실행 전 문법 검사 단계에서 나므로, 앞부분 코드도 한 줄도 실행되지 않음
- 인자를 여러 줄로 나눠 쓸 때 생기기 쉬운 실수임

## 5. 학습 시간 측정 (`time.time()`)

`keras18_time.py`

```python
import time

start_time = time.time()        # 현재 시각 (시작)
model.fit(x_train, y_train, epochs=2, batch_size=8, validation_split=0.2)
end_time = time.time()          # 현재 시각 (끝)

print("걸린 시간: ", round(end_time - start_time, 2), " 초")   # 3.9 초
```

**`time.time()`의 동작**
- 1970년부터 지금까지의 초를 실수로 반환함
- 값 자체는 의미가 없고 두 시각의 차이를 봄
- 괄호를 빠뜨리면 함수 객체가 담겨 뺄셈에서 `TypeError`가 남

### 5-1. 학습 시간을 재는 이유

하이퍼파라미터가 성능뿐 아니라 시간에도 영향을 주기 때문임

| 바꾸는 값 | 시간 |
|---|---|
| `epochs` ↑ | 비례해서 증가 |
| `batch_size` ↓ | 크게 증가 (갱신 횟수가 늘어남) |
| 층·노드 ↑ | 증가 |

- 캐글 자전거는 `batch_size=8`이라 한 epoch에 갱신이 871번, 500 epoch이면 43만 번
- "R²를 0.01 올리려고 시간을 10배 쓰는 게 맞나"를 판단하려면 숫자가 있어야 함

**규칙**
- `start_time`은 `fit` 바로 앞에 둠. 데이터 로딩까지 포함하면 다른 것을 재게 됨

## 6. history 시각화

**시각화가 필요한 이유**
- `val_loss`를 로그로 읽는 데는 한계가 있음. 500줄을 훑어 어디서 꺾였는지 찾기 어려움

```python
hist = model.fit(x_train, y_train, epochs=100, batch_size=32, validation_split=0.2)

print(hist)          # <keras.src.callbacks.history.History object at 0x...>
print(hist.history)  # {'loss': [5.20, 1.16, ...], 'val_loss': [1.06, 1.20, ...]}
```

- `fit`은 History 객체를 돌려주고, 여기에 매 epoch의 loss가 기록되어 있음
- `validation_split`을 주지 않으면 `val_loss` 키 자체가 없음

### 6-1. 그래프 그리기

```python
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))                                      # 도화지 크기 (인치)
plt.plot(hist.history['loss'],     c='red',  label='loss')      # x 생략 시 0,1,2... 자동
plt.plot(hist.history['val_loss'], c='blue', label='val_loss')
plt.legend(loc='upper right')
plt.title('California Loss')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.grid()
plt.show()
```

**그래프 읽는 법**
1. 두 선이 나란히 내려감 → 정상
2. loss는 내려가는데 val_loss가 위로 벌어짐 → 과적합 (벌어지는 epoch이 멈출 지점)
3. 두 선이 높은 채 평평함 → 과소적합
4. val_loss가 심하게 요동침 → 검증셋이 작거나 학습률이 큼

### 6-2. cf) 초반 큰 loss 자르기

- 첫 몇 epoch의 loss는 훨씬 큼 (캘리포니아는 `5.20` → 이후 `0.8` 수준)
- 그래프의 눈금이 초반 값에 맞춰져 이후 변화가 잘 보이지 않음

```python
plt.plot(hist.history['loss'][3:],     c='red',  label='loss')      # 앞 3개 버리기
plt.plot(hist.history['val_loss'][3:], c='blue', label='val_loss')
```

- x축 눈금이 0부터 다시 시작하므로 실제 epoch 번호와 3만큼 어긋남

### 6-3. cf) 한글 깨짐과 폰트 설정 위치

폰트 설정이 `plt.figure()` 뒤에 오면 한글이 □□□로 깨짐

| 설정 위치 | 한글 | 원인 |
|---|---|---|
| `plt.figure()` 뒤 | 깨짐 | `figure()`/`plot()` 시점에 기본 폰트로 이미 초기화됨 |
| `plt.figure()` 앞 | 정상 | 새로 만들어지는 요소에 새 폰트가 적용됨 |
| 설정 없음 | 깨짐 | 영문 폰트에 한글 글리프가 없음 |

```python
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호 깨짐 방지
```

- `import` 직후 맨 위에 두는 것이 가장 안전함

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

x = np.array(range(1, 21))
y = np.array(range(1, 21))

model = Sequential([Dense(8, input_dim=1), Dense(4), Dense(1)])
model.compile(loss='mse', optimizer='adam')

hist = model.fit(x, y, epochs=10, batch_size=4,
                 validation_split=0.2, verbose=0)   # 반환값을 hist로 받음

print(type(hist))                 # <class 'keras.src.callbacks.history.History'>
print(hist.history.keys())        # dict_keys(['loss', 'val_loss'])
print(len(hist.history['loss']))  # 10  <- epochs와 같은 길이

# validation을 안 주면 val_loss 키가 아예 없음
hist2 = model.fit(x, y, epochs=10, batch_size=4, verbose=0)
print(hist2.history.keys())       # dict_keys(['loss'])

## 7. EarlyStopping (조기 종료)

`val_loss`가 더 이상 좋아지지 않으면 훈련을 자동으로 멈추는 케라스 콜백 (`keras20_EarlyStopping1~5`)

**EarlyStopping이 필요한 이유**
- §6에서 본 것처럼 `val_loss`가 최솟값을 찍고 벌어지기 시작하는 지점에서 멈춰야 함
- 사람이 500, 1000 epoch 로그를 지켜보다 직접 끊을 수는 없음

```python
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor='val_loss',         # 감시할 기준 지표
    mode='min',                 # 최솟값을 추적 (auto, min, max)
    patience=10,                # 갱신이 안 되어도 10번은 더 지켜봄
    restore_best_weights=True,  # 가장 좋았던 순간의 가중치로 복원
)

hist = model.fit(x_train, y_train, epochs=500, batch_size=32,
                 validation_split=0.2,
                 callbacks=[es],        # 리스트 형태로 전달
                 )
```

### 7-1. 주요 파라미터

1. `monitor` : 감시할 지표 (기본값 `'val_loss'`)
   - `loss`는 외워도 계속 떨어지므로 `val_loss`로만 과적합을 판단할 수 있음
2. `mode` : 좋은 방향 (기본값 `'auto'`)
   - `'min'` 낮을수록 좋은 지표 / `'max'` 높을수록 좋은 지표 / `'auto'` 이름을 보고 판단
3. `patience` : 갱신 없이 몇 번을 참을지 (기본값 `0`)
   - 손실은 오르내리며 요동치므로 여유가 필요함
4. `restore_best_weights` : 가장 좋았던 가중치로 되돌릴지 (기본값 `False`)

### 7-2. restore_best_weights

`patience=10`이면 최저점을 찍은 뒤 10번 더 개선되지 않을 때 학습이 끝남

```
Epoch 35 : val_loss 0.5200  <-- 역대 최저점
Epoch 36 : val_loss 0.5310  (1회 실패)
...
Epoch 45 : val_loss 0.6150  (10회 실패 -> 중단)
```

| 설정 | 남는 가중치 |
|---|---|
| `False` (기본값) | Epoch 45 → 최저점에서 10번 후퇴한 상태 |
| `True` | Epoch 35 → 가장 좋았던 시점으로 되돌림 |

- 멈추는 것은 10번 더 확인하고 멈추더라도, 쓸 가중치는 최저점으로 되돌려야 함

### 7-3. callbacks를 리스트로 넘기는 이유

```python
callbacks=[es]
```

- 콜백은 `EarlyStopping` 외에도 가중치를 저장하는 `ModelCheckpoint`, 학습률을 낮추는 `ReduceLROnPlateau` 등 여러 개를 동시에 걸 수 있게 설계됨
- 그래서 하나만 넘겨도 리스트 `[ ]`로 감싸야 함

### 7-4. patience 정하기

`patience`가 `epochs`에 비해 크면 EarlyStopping을 단 의미가 사라짐

| 파일 | epochs | patience | 비중 |
|---|---|---|---|
| keras20_2 당뇨 | 640 | 200 → 64 | 31% → 10% |
| keras20_4 따릉이 | 500 | 200 → 50 | 40% → 10% |
| keras20_5 캐글자전거 | 200 | 100 → 20 | 50% → 10% |

실제로 돌려본 결과

```
patience 200 / epochs 640  ->  실제 학습 [338, 640]   ES 발동 1/2회
patience  20 / epochs 1000 ->  실제 학습 [ 27,  31]   ES 발동 2/2회
```

**규칙**
1. `patience`가 epochs의 절반이면 최소 절반은 항상 돌고 대부분 끝까지 감
2. 반대로 너무 작으면 손실이 한 번 튄 것만으로 멈춰 과소적합이 됨
3. 기준이 없으면 epochs의 10~20%에서 시작해 그래프를 보며 조정함